# Bronze Ingestion — Mapa Digital do Fluxo do Paciente

**Objetivo:** Ingerir arquivos CSV anonimizados do Volume `landing_zone` e materializar como tabelas Delta na camada Bronze.

**Input:** `/Volumes/hospital_santa_rosa/bronze_fluxo/landing_zone/*.csv`  
**Output:** Tabelas `bronze_*_raw` no schema `hospital_santa_rosa.bronze_fluxo`  
**Método:** Auto Loader (`cloudFiles`) com checkpoint por tabela  
**Cadência:** Execução manual mensal (Fase 1)

In [0]:
# Configuração: mapeamento do arquivo -> tabela bronze
BRONZE_TABLES = {
    'altas': 'bronze_altas_raw',
    'atendimento_emergencia': 'bronze_atendimento_emergencia_raw',
    'cirurgias': 'bronze_cirurgias_raw',
    'epidemio': 'bronze_epidemio_raw',
    'exames_imagem': 'bronze_exames_imagem_raw',
    'exames_laboratoriais': 'bronze_exames_laboratoriais_raw',
    'movimentacoes': 'bronze_movimentacoes_raw',
    'internacoes': 'bronze_internacoes_raw',
}

VOLUME_PATH = '/Volumes/hospital_santa_rosa/bronze_fluxo/landing_zone'
CHECKPOINT_BASE = '/Volumes/hospital_santa_rosa/bronze_fluxo/landing_zone/_checkpoints'
CATALOG_SCHEMA = 'hospital_santa_rosa.bronze_fluxo'

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def ingest_to_bronze(folder_name: str, table_name: str):
    """Ingere um arquivo CSV do Volume para uma tabela Bronze via Auto Loader
    Args:
        folder_name: Nome da pasta onde o arquivo CSV esta no Volume
        table_name: Nome da tabela Bronze de destino
    """
    # Monta o caminho do arquivo
    folder_path = f'{VOLUME_PATH}/{folder_name}'
    # Caminho onde Auto Loader salva o checkpoint de cada tabela
    checkpoint_path = f'{CHECKPOINT_BASE}/{table_name}'
    # Monta o nome completo da tabela no namespace do Unity Catalog
    full_table_name = f'{CATALOG_SCHEMA}.{table_name}'

    # Leitura do arquivo via Auto Loader (streaming incremental)
    df = (
        spark.readStream
        .format('cloudFiles') # identifica que usa Auto Loader
        .option('cloudFiles.format', 'csv') # informa que os arquivos são CSV
        .option('header', 'True') # informa que o arquivo tem cabeçalho
        .option('sep', ';') # informa que o separador é ;
        .option('cloudFiles.schemaLocation', f'{CHECKPOINT_BASE}/{table_name}_schema') # local onde Auto Loader salva o schema
        .load(folder_path) # caminho do arquivo para leitura
    )

    # Adiciona colunas de metadados de ingestão antes de gravar na bronze
    df = (
        df
        .selectExpr('*', '_metadata.file_path as _source_file')
        .withColumn('_ingestion_timestamp', current_timestamp())
    )

    # Materializa os dados na tabela bronze
    (
        df.writeStream # inicia a escrita streaming
        .format('delta') # grava no formato Delta Lake
        .option('checkpointLocation', checkpoint_path) # checkpoint para garantir que não haja duplicidade
        .option('mergeSchema', 'true')
        .outputMode('append') # modo de gravação
        .trigger(availableNow=True) # processa todos os arquivos disponíveis
        .toTable(full_table_name) # tabela de destino
    )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS hospital_santa_rosa.bronze_fluxo.bronze_cirurgias_raw
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);
CREATE TABLE IF NOT EXISTS hospital_santa_rosa.bronze_fluxo.bronze_epidemio_raw
TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5');

CREATE TABLE IF NOT EXISTS hospital_santa_rosa.bronze_fluxo.bronze_exames_imagem_raw
TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5');

CREATE TABLE IF NOT EXISTS hospital_santa_rosa.bronze_fluxo.bronze_internacoes_raw
TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5');

In [0]:
# Percorre o dicionário e chama a função de ingestão para cada arquivo/tabela
for folder_name, table_name in BRONZE_TABLES.items():
    print(f'Iniciando ingestão: {folder_name} -> {table_name}')
    ingest_to_bronze(folder_name, table_name)

In [0]:
%sql
SELECT *
FROM hospital_santa_rosa.bronze_fluxo.bronze_altas_raw
WHERE ATENDIMENTO = '6a969136834f71fa98a208098092f7131ec4e4d3d75f1bf456e2754a1616ca94'